# Lesson 1 - System info and battery

This notebook reads basic system information from the Jetson Nano and the battery
voltage from the on-board INA219 power sensor, then visualizes the voltage over time.

The control helpers live in `jbot.py` in the parent folder, so the first cell adds
that folder to the import path.

In [ ]:
import sys
sys.path.insert(0, "..")
import subprocess

def read_text(path):
    try:
        return open(path).read().strip().strip("\x00")
    except Exception:
        return "n/a"

print("Model   :", read_text("/proc/device-tree/model"))
print("L4T     :", read_text("/etc/nv_tegra_release").split(",")[0])
print("Kernel  :", subprocess.check_output(["uname", "-r"]).decode().strip())

## Read the battery once

`read_battery()` reads the bus-voltage register of the INA219 at I2C address `0x41`
and returns volts. A healthy 3S pack reads about 11-12.6 V.

In [ ]:
from jbot import read_battery

voltage = read_battery()
print("%.2f V" % voltage)

## Estimate the charge level

A rough state-of-charge estimate maps 9.0 V (empty) to 12.6 V (full).

In [ ]:
def charge_percent(v):
    return max(0.0, min(100.0, (v - 9.0) / (12.6 - 9.0) * 100.0))

print("%.0f %%" % charge_percent(read_battery()))

## Live voltage plot

Sample the battery a few times and plot it. Run this while driving the motors to
see the voltage sag under load.

In [ ]:
%matplotlib inline
import time
import matplotlib.pyplot as plt
from IPython import display

xs, ys = [], []
for i in range(20):
    ys.append(read_battery())
    xs.append(i)
    plt.clf()
    plt.plot(xs, ys, "-o", color="tab:green")
    plt.ylim(9, 13)
    plt.xlabel("sample")
    plt.ylabel("Voltage (V)")
    plt.title("Battery voltage")
    plt.grid(True, alpha=0.3)
    display.clear_output(wait=True)
    display.display(plt.gcf())
    time.sleep(0.5)